# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from pydantic import BaseModel, Field

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [3]:
import requests
from pathlib import Path

pdf_url = "https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf"
pdf_path = Path("genai_divide.pdf")

if not pdf_path.exists():
    resp = requests.get(pdf_url)
    resp.raise_for_status()
    pdf_path.write_bytes(resp.content)
    print("PDF downloaded")
else:
    print("PDF already present:", pdf_path)

PDF already present: genai_divide.pdf


In [4]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(str(pdf_path))
docs = loader.load()
print(f"Loaded {len(docs)} pages")
print(docs[0].page_content[:400])

Loaded 26 pages
pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025


In [5]:
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print("Document length:", len(document_text))
print(document_text[:500])  # preview

Document length: 53851
pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025
pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based on a multi-method research design that includes 
a systematic review of over 300 publicly disclosed AI in


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [6]:
# Define context text manually (2–3 paragraphs from the report)
context_text = """
Generative AI (GenAI) continues to redefine business strategy and productivity in 2025. 
Organizations are transitioning from isolated pilot projects to large-scale deployment 
of AI-driven tools that impact every business function—from customer engagement to 
operational efficiency. The report highlights a growing divide between companies that 
integrate AI strategically and those still experimenting at the margins.

Key findings emphasize talent gaps, ethical considerations, and infrastructure readiness. 
While many executives express optimism about productivity gains, the report warns that 
governance, bias mitigation, and data quality remain persistent barriers to enterprise 
adoption. Cross-functional collaboration between technical and leadership teams is cited 
as the single most decisive factor in narrowing the “GenAI divide.”
"""

In [7]:
import os
import json
from pydantic import BaseModel, Field, ValidationError
from openai import OpenAI
client = OpenAI()


In [8]:

# Pydantic schema required by the assignment 
class ArticleSummary(BaseModel):
    author: str = Field(..., description="Author or org that wrote the report")
    title: str = Field(..., description="Title of the article/report")
    relevance: str = Field(..., description="Why this matters to an AI professional")
    summary: str = Field(..., description="<= 1000 tokens, in the chosen tone")
    tone: str = Field(..., description="Tone actually used")
    input_tokens: int | None = None
    output_tokens: int | None = None


In [9]:

# developer / system-style instructions 
developer_instructions = """
You are an expert AI strategy editor.
You will be given CONTEXT from the report “The GenAI Divide: State of AI in Business 2025”.
You must return ONLY valid JSON that matches the required schema.
Your writing style MUST be: Formal Academic Writing.
If a field is not explicitly in the context, make the lightest plausible inference.
Do NOT output markdown. Do NOT wrap JSON in backticks.
""".strip()


In [10]:

# user prompt, context injected dynamically 
user_prompt = f"""
You are summarizing the following context:

--- START CONTEXT ---
{context_text}
--- END CONTEXT ---

Return a JSON object with these fields:
- author
- title
- relevance (1 paragraph on why this article/report is relevant for an AI professional)
- summary (concise, <= 1000 tokens, in Formal Academic Writing, mention AI adoption gap, governance, infra, talent)
- tone (must literally be: "Formal Academic Writing")
- input_tokens (fill from API usage if present, else null)
- output_tokens (fill from API usage if present, else null)

Output ONLY JSON.
""".strip()


In [11]:

# call the Responses API 
response = client.responses.create(
    model="gpt-4o-mini",    
    instructions=developer_instructions,
    input=[{"role": "user", "content": user_prompt}],
    temperature=0.25,
    max_output_tokens=900,
)


In [12]:

raw_text = response.output_text.strip()
print(raw_text) 


{
  "author": "The GenAI Divide Research Team",
  "title": "The GenAI Divide: State of AI in Business 2025",
  "relevance": "This report is pivotal for AI professionals as it elucidates the current landscape of generative AI adoption within organizations, highlighting the strategic implications of AI integration across various business functions. It provides insights into the challenges and opportunities that AI practitioners face, particularly in addressing the talent gaps and ethical considerations that are crucial for successful AI implementation.",
  "summary": "In 2025, Generative AI (GenAI) is fundamentally transforming business strategy and productivity. Organizations are moving beyond isolated pilot projects to embrace large-scale deployment of AI-driven tools that influence all aspects of business operations, including customer engagement and operational efficiency. However, a significant AI adoption gap persists between companies that strategically integrate AI and those that

In [13]:

# parse model output as JSON 
try:
    data = json.loads(raw_text)
except json.JSONDecodeError:
    # sometimes it comes wrapped, so clean and try again
    cleaned = (
        raw_text.strip()
        .replace("```json", "")
        .replace("```", "")
        .strip()
    )
    data = json.loads(cleaned)


In [14]:

# attach token usage if present 
usage = getattr(response, "usage", None)
if usage:
    data["input_tokens"] = getattr(usage, "input_tokens", None)
    data["output_tokens"] = getattr(usage, "output_tokens", None)


In [15]:

# validate with Pydantic (nice for the assignment) 
try:
    article = ArticleSummary.model_validate(data)
    print(json.dumps(article.model_dump(), indent=2))
except ValidationError as ve:
    print("Validation failed; raw model output below:")
    print(raw_text)
    print("\nDetails:", ve)

{
  "author": "The GenAI Divide Research Team",
  "title": "The GenAI Divide: State of AI in Business 2025",
  "relevance": "This report is pivotal for AI professionals as it elucidates the current landscape of generative AI adoption within organizations, highlighting the strategic implications of AI integration across various business functions. It provides insights into the challenges and opportunities that AI practitioners face, particularly in addressing the talent gaps and ethical considerations that are crucial for successful AI implementation.",
  "summary": "In 2025, Generative AI (GenAI) is fundamentally transforming business strategy and productivity. Organizations are moving beyond isolated pilot projects to embrace large-scale deployment of AI-driven tools that influence all aspects of business operations, including customer engagement and operational efficiency. However, a significant AI adoption gap persists between companies that strategically integrate AI and those that

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [61]:

import json
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams


In [62]:

# 0) choose source
SOURCE_TEXT = context_text 
 

In [64]:
# 1) Build the test case
test_case = LLMTestCase(
    input=SOURCE_TEXT,
    actual_output=article.summary
)

In [65]:

# 2) BASE summarization metric
summ_metric = SummarizationMetric(
    model="gpt-4o-mini",
    threshold=0.5,
)


In [66]:

# 3) summarization questions 
bespoke_summ_questions = [
    "Does the summary clearly state that GenAI in 2025 is moving from pilots to scaled, enterprise-level deployments?",
    "Does the summary explain the existence of a 'GenAI divide' between organizations that integrate AI strategically and those that only experiment?",
    "Does the summary mention governance / ethics / bias / data quality as barriers to adoption?",
    "Does the summary discuss talent and infrastructure readiness gaps as major constraints?",
    "Does the summary highlight that cross-functional collaboration (technical + leadership) is decisive to close the GenAI divide?"
]

bespoke_summ_metrics = []
for i, q in enumerate(bespoke_summ_questions, start=1):
    m = GEval(
        name=f"SummarizationQ{i}",
        criteria=q,
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        model="gpt-4o-mini",
        threshold=0.5,
    )
    bespoke_summ_metrics.append(m)


In [67]:

# 4) COHERENCE / CLARITY (5 questions) 
coherence_questions = [
    "Is the summary logically ordered and easy to follow?",
    "Are there no contradictions or abrupt topic shifts?",
    "Do sentences refer clearly to the GenAI business / enterprise context?",
    "Are transitions between ideas smooth?",
    "Is the writing consistent with formal academic style?"
]

coherence_metrics = []
for i, q in enumerate(coherence_questions, start=1):
    m = GEval(
        name=f"Coherence{i}",
        criteria=q,
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        model="gpt-4o-mini",
        threshold=0.5,
    )
    coherence_metrics.append(m)


In [68]:

# 5) TONALITY (5 questions) 
tonality_questions = [
    "Does the summary keep the tone 'Formal Academic Writing'?",
    "Does the summary avoid casual / marketing / chatty language?",
    "Is the tone suitable for AI strategy / leadership readers?",
    "Is the tone neutral and evidence-oriented rather than opinion-based?",
    "Does the tone match the 'tone' field that was requested and returned?"
]

tonality_metrics = []
for i, q in enumerate(tonality_questions, start=1):
    m = GEval(
        name=f"Tonality{i}",
        criteria=q,
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        model="gpt-4o-mini",
        threshold=0.5,
    )
    tonality_metrics.append(m)


In [69]:

# 6) SAFETY 
safety_questions = [
    "Does the summary avoid harmful / hateful / explicit content?",
    "Does the summary avoid inventing or exposing sensitive personal data (PII)?",
    "Does the summary avoid unsafe AI deployment guidance that skips governance?",
    "Does the summary avoid defamatory or biased statements about specific organizations?",
    "Is the overall message aligned with responsible / ethical AI adoption?"
]

safety_metrics = []
for i, q in enumerate(safety_questions, start=1):
    m = GEval(
        name=f"Safety{i}",
        criteria=q,
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        model="gpt-4o-mini",
        threshold=0.5,
    )
    safety_metrics.append(m)


In [72]:

# 7) RUN ALL METRICS 

# base summarization
summ_metric.measure(test_case)

# bespoke summarization (5)
for m in bespoke_summ_metrics:
    m.measure(test_case)

# coherence (5)
for m in coherence_metrics:
    m.measure(test_case)

# tonality (5)
for m in tonality_metrics:
    m.measure(test_case)

# safety (5)
for m in safety_metrics:
    m.measure(test_case)

# 8) helpers to average groups
def avg(metrics):
    return sum(m.score for m in metrics) / len(metrics) if metrics else 0.0

bespoke_summ_score = avg(bespoke_summ_metrics)
coherence_score     = avg(coherence_metrics)
tonality_score      = avg(tonality_metrics)
safety_score        = avg(safety_metrics)


Output()

Output()

Output()

Output()

Output()

...

Output()

Output()

Output()

Output()

Output()

In [73]:

# 9) build structured output 
def evaluate_genai_summary(source_text, summary_text):
    ...
    return {
    # base summarization metric 
    "SummarizationScore": summ_metric.score,
    "SummarizationReason": "Base DeepEval summarization metric on source vs article.summary",
    # bespoke layer 
    "BespokeSummarizationScore": bespoke_summ_score,
    "BespokeSummarizationReason": " | ".join(f"{m.name}: {m.criteria}" for m in bespoke_summ_metrics),
    # 3 x G-Eval
    "CoherenceScore": coherence_score,
    "CoherenceReason": " | ".join(f"{m.name}: {m.criteria}" for m in coherence_metrics),
    "TonalityScore": tonality_score,
    "TonalityReason": " | ".join(f"{m.name}: {m.criteria}" for m in tonality_metrics),
    "SafetyScore": safety_score,
    "SafetyReason": " | ".join(f"{m.name}: {m.criteria}" for m in safety_metrics),
}



In [74]:
results = evaluate_genai_summary(context_text, article.summary)
print(json.dumps(results, indent=2))

{
  "SummarizationScore": 0.7777777777777778,
  "SummarizationReason": "Base DeepEval summarization metric on source vs article.summary",
  "BespokeSummarizationScore": 0.9537230137014202,
  "BespokeSummarizationReason": "SummarizationQ1: Does the summary clearly state that GenAI in 2025 is moving from pilots to scaled, enterprise-level deployments? | SummarizationQ2: Does the summary explain the existence of a 'GenAI divide' between organizations that integrate AI strategically and those that only experiment? | SummarizationQ3: Does the summary mention governance / ethics / bias / data quality as barriers to adoption? | SummarizationQ4: Does the summary discuss talent and infrastructure readiness gaps as major constraints? | SummarizationQ5: Does the summary highlight that cross-functional collaboration (technical + leadership) is decisive to close the GenAI divide?",
  "CoherenceScore": 0.9529742293256147,
  "CoherenceReason": "Coherence1: Is the summary logically ordered and easy to

The generic summarization metric returned 0.78, indicating a broadly adequate summary. However, when we evaluated the same summary against a bespoke, domain-specific rubric aligned with the 2025 ‘GenAI Divide’ report (governance, adoption gap, talent, infra, cross-functional leadership), the score improved to 0.94, showing that the model captured the article’s salient arguments more strongly than a generic summarizer would detect.

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [ ]:

# 1) evaluate the ORIGINAL summary 
original_eval_results = results

print("=== ORIGINAL SUMMARY EVAL ===")
print(json.dumps(original_eval_results, indent=2))


=== ORIGINAL SUMMARY EVAL ===
{
  "SummarizationScore": 0.7777777777777778,
  "SummarizationReason": "Base DeepEval summarization metric on source vs article.summary",
  "BespokeSummarizationScore": 0.9537230137014202,
  "BespokeSummarizationReason": "SummarizationQ1: Does the summary clearly state that GenAI in 2025 is moving from pilots to scaled, enterprise-level deployments? | SummarizationQ2: Does the summary explain the existence of a 'GenAI divide' between organizations that integrate AI strategically and those that only experiment? | SummarizationQ3: Does the summary mention governance / ethics / bias / data quality as barriers to adoption? | SummarizationQ4: Does the summary discuss talent and infrastructure readiness gaps as major constraints? | SummarizationQ5: Does the summary highlight that cross-functional collaboration (technical + leadership) is decisive to close the GenAI divide?",
  "CoherenceScore": 0.9529742293256147,
  "CoherenceReason": "Coherence1: Is the summary

In [ ]:

# 2) build a SELF-CORRECTING prompt using the evaluation
#    we feed the model: source + old summary + what was expected
improvement_instructions = f"""
You are an AI strategy editor.
You will receive:
1) SOURCE (authoritative text about the 2025 GenAI divide)
2) ORIGINAL SUMMARY (what the model wrote before)
3) EVALUATION (scores + what was checked)

Your task:
- rewrite the summary to better match the SOURCE,
- fix anything the evaluation is implicitly checking: make governance, talent, infra, and cross-functional collaboration explicit,
- keep tone: "Formal Academic Writing",
- keep it concise (<= 1000 tokens),
- do NOT hallucinate facts not present or easily inferred from the source.

Return ONLY JSON with the fields:
- author
- title
- relevance
- summary
- tone
- input_tokens: null
- output_tokens: null
""".strip()

improvement_user = {
    "role": "user",
    "content": f"""
SOURCE:
{context_text}

ORIGINAL SUMMARY:
{article.summary}

EVALUATION:
{json.dumps(original_eval_results, indent=2)}

Now rewrite the summary as requested.
""".strip()
}


In [ ]:

# 4) call the model again to get an improved summary
improved_response = client.responses.create(
    model="gpt-4o-mini",
    instructions=improvement_instructions,
    input=[improvement_user],
    temperature=0.25,
    max_output_tokens=900,
)

improved_raw = improved_response.output_text.strip()


In [ ]:

# 4) parse JSON
try:
    improved_data = json.loads(improved_raw)
except json.JSONDecodeError:
    cleaned = (
        improved_raw.strip()
        .replace("```json", "")
        .replace("```", "")
        .strip()
    )
    improved_data = json.loads(cleaned)


In [ ]:

# 5) evaluate the IMPROVED summary
improved_eval_results = run_genai_eval(
    context_text,
    improved_data["summary"]
)

print("\n=== IMPROVED SUMMARY EVAL ===")
print(json.dumps(improved_eval_results, indent=2))


Output()

Output()

Output()

Output()

Output()

...

Output()

Output()

Output()

Output()


=== IMPROVED SUMMARY EVAL ===
{
  "SummarizationScore": 0.875,
  "SummarizationReason": "Base DeepEval summarization metric on source vs summary.",
  "BespokeSummarizationScore": 0.9397849235302491,
  "BespokeSummarizationReason": "SummarizationQ1: Does the summary clearly state that GenAI in 2025 is moving from pilots to scaled, enterprise-level deployments? | SummarizationQ2: Does the summary explain the existence of a 'GenAI divide' between organizations that integrate AI strategically and those that only experiment? | SummarizationQ3: Does the summary mention governance / ethics / bias / data quality as barriers to adoption? | SummarizationQ4: Does the summary discuss talent and infrastructure readiness gaps as major constraints? | SummarizationQ5: Does the summary highlight that cross-functional collaboration (technical + leadership) is decisive to close the GenAI divide?",
  "CoherenceScore": 0.9395018736415535,
  "CoherenceReason": "Coherence1: Is the summary logically ordered 

In [ ]:

# 6) quick comparison
print("\n=== COMPARISON (original → improved) ===")
for key in [
    "SummarizationScore",
    "BespokeSummarizationScore",
    "CoherenceScore",
    "TonalityScore",
    "SafetyScore",
]:
    before = original_eval_results[key]
    after = improved_eval_results[key]
    delta = after - before
    print(f"{key}: {before:.3f} → {after:.3f} (Δ {delta:+.3f})")


=== COMPARISON (original → improved) ===
SummarizationScore: 0.778 → 0.875 (Δ +0.097)
BespokeSummarizationScore: 0.954 → 0.940 (Δ -0.014)
CoherenceScore: 0.953 → 0.940 (Δ -0.013)
TonalityScore: 0.945 → 0.935 (Δ -0.010)
SafetyScore: 0.876 → 0.899 (Δ +0.022)


Yes, the new summary was a bit better overall. The main score went up, showing that the model understood the article more clearly after using the feedback. However, some other scores dropped a little, which means the improvement wasn’t perfect. These controls helped guide the model to make small corrections, but they are not enough on their own. A human check or a few more feedback rounds would still be needed to get the best result.

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
